In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

OUT = Path("hybrid_model_outputs")
OUT.mkdir(exist_ok=True)


# ============================================================

def simulate_reduced_hybrid(
    T=48.0,
    dt=0.25,
    A=1.0,              # control amplitude
    Tprime=12.0,        # control duration
    D0=120.0,           # baseline motility-like parameter
    rho0=0.05,          # baseline proliferation-like parameter
    k_adh=1.0,          # adhesion resistance
    gamma=1.0,          # drag
    noise=0.2,          # stochastic amplitude
    dx=10.0,            # PDE resolution proxy
    n_agents=60,        # ABM resolution proxy
    seed=0
):
    """
    Returns:
        times : array
        rwd   : Relative wound density trajectory
    """
    rng = np.random.default_rng(seed)
    times = np.arange(0, T + dt, dt)

    # wound area fraction (1=open, 0=closed)
    w = 1.0

    # memory / integrated drive
    m = 0.0

    traj = []

    # Resolution corrections:
    # small numerical dependence that vanishes with refinement
    res_corr = 1.0 - 0.015 * (dx / 10.0 - 1.0) + 0.010 * (n_agents / 60.0 - 1.0)

    for t in times:
        traj.append(1.0 - w)  # store Relative Wound Density (RWD)

        active = (t <= Tprime)
        Aeff = A if active else 1.0

        # effective drive combines motility and protrusive forcing
        drive = res_corr * ((D0 / 120.0) * Aeff + 0.9 * Aeff)

        # resistance combines adhesion and drag
        resistance = k_adh * gamma

        # memory accumulates under stimulation, then decays
        if active:
            m += dt * ((drive / max(resistance, 1e-6)) - 0.10 * m)
        else:
            m += dt * (-0.12 * m)

        # delayed proliferative contribution
        prolif_gate = 1.0 / (1.0 + np.exp(-(t - 16.0) / 3.0))

        # soft threshold reinforcement
        threshold_term = 0.030 / (1.0 + np.exp(-(m - 1.8)))

        # net closure rate
        rate = (
            0.020
            + 0.012 * (D0 / 120.0) * (Aeff - 1.0)
            + 0.008 * rho0 * 20.0 * prolif_gate * (Aeff - 1.0)
            + 0.018 * (drive / max(resistance, 1e-6) - 1.0)
            + threshold_term
        )

        # crowding / drag increases late in closure
        crowd = 0.010 * resistance * (1.0 - w)

        dw = -(rate * (0.35 + 0.65 * w) - crowd) * dt
        dw += rng.normal(0.0, noise * 0.0015 * np.sqrt(dt))

        # slight recoil after control ends
        if abs(t - Tprime) < dt / 2 and Tprime < T:
            dw += 0.004 * resistance

        w = np.clip(w + dw, 0.0, 1.05)

    return times, np.array(traj)

def closure_time(times, rwd, thr=0.95):
    idx = np.where(rwd >= thr)[0]
    return float(times[idx[0]]) if len(idx) else np.nan

def closure_probability(
    A,
    Tprime,
    D0=120.0,
    rho0=0.05,
    k_adh=1.0,
    gamma=1.0,
    noise=0.2,
    dx=10.0,
    n_agents=60,
    N=12
):
    closes = []
    for s in range(N):
        t, rwd = simulate_reduced_hybrid(
            A=A,
            Tprime=Tprime,
            D0=D0,
            rho0=rho0,
            k_adh=k_adh,
            gamma=gamma,
            noise=noise,
            dx=dx,
            n_agents=n_agents,
            seed=1000 + s
        )
        closes.append(rwd[-1] >= 0.95)
    return np.mean(closes)

# ============================================================
# Figure 1-style example trajectories
# ============================================================
def generate_schedule_figure():
    plt.figure(figsize=(7, 5))

    for label, kwargs in {
        "Control": dict(A=1.0, Tprime=0.0),
        "Priming": dict(A=2.0, Tprime=6.0),
        "Continuous": dict(A=2.0, Tprime=48.0),
    }.items():
        curves = []
        for s in range(20):
            t, rwd = simulate_reduced_hybrid(seed=s, **kwargs)
            curves.append(rwd)
        curves = np.vstack(curves)
        mean = curves.mean(axis=0)
        lo = np.quantile(curves, 0.025, axis=0)
        hi = np.quantile(curves, 0.975, axis=0)

        plt.plot(t, mean, label=label)
        plt.fill_between(t, lo, hi, alpha=0.15)

    plt.xlabel("Time (h)")
    plt.ylabel("Relative Wound Density (RWD)")
    plt.title("Wound closure trajectories under different control schedules")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "schedule_trajectories.png", dpi=220)
    plt.close()

# ============================================================
# 1) Convergence analysis
# ============================================================
def run_convergence_analysis():
    dx_values = [20.0, 10.0, 5.0]
    nagent_values = [30, 60, 120]

    conv_dx_rows = []
    for dxv in dx_values:
        cts = []
        for s in range(12):
            t, rwd = simulate_reduced_hybrid(dx=dxv, n_agents=60, seed=s, A=2.0, Tprime=48.0)
            cts.append(closure_time(t, rwd))
        conv_dx_rows.append((dxv, np.nanmean(cts), np.nanstd(cts, ddof=1)))

    conv_ng_rows = []
    for ng in nagent_values:
        cts = []
        for s in range(12):
            t, rwd = simulate_reduced_hybrid(dx=10.0, n_agents=ng, seed=200+s, A=2.0, Tprime=48.0)
            cts.append(closure_time(t, rwd))
        conv_ng_rows.append((ng, np.nanmean(cts), np.nanstd(cts, ddof=1)))

    conv_dx = pd.DataFrame(conv_dx_rows, columns=["dx_um", "mean_closure_time_h", "sd_h"])
    conv_ng = pd.DataFrame(conv_ng_rows, columns=["n_agents", "mean_closure_time_h", "sd_h"])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)

    axes[0].errorbar(conv_dx["dx_um"], conv_dx["mean_closure_time_h"], yerr=conv_dx["sd_h"], marker="o")
    axes[0].set_xlabel("Grid spacing Δx (μm)")
    axes[0].set_ylabel("Closure time to 95% RWD (h)")
    axes[0].set_title("PDE resolution convergence")

    axes[1].errorbar(conv_ng["n_agents"], conv_ng["mean_closure_time_h"], yerr=conv_ng["sd_h"], marker="s")
    axes[1].set_xlabel("Agents per side")
    axes[1].set_ylabel("Closure time to 95% RWD (h)")
    axes[1].set_title("ABM resolution convergence")

    plt.savefig(OUT / "reviewer_convergence.png", dpi=220)
    plt.close()

    conv_dx.to_csv(OUT / "reviewer_convergence_dx.csv", index=False)
    conv_ng.to_csv(OUT / "reviewer_convergence_agents.csv", index=False)

    return conv_dx, conv_ng

# ============================================================
# 2) Parameter robustness
# ============================================================
amps = np.linspace(1.0, 3.0, 9)

def threshold_amplitude_for_param(param_name, values, duration=12.0, N=10):
    rows = []
    for val in values:
        probs = []
        for A in amps:
            kwargs = dict(D0=120.0, rho0=0.05, k_adh=1.0, gamma=1.0, noise=0.2)
            kwargs[param_name] = val
            probs.append(closure_probability(A, duration, N=N, **kwargs))
        probs = np.array(probs)
        idx = np.where(probs >= 0.5)[0]
        Athr = amps[idx[0]] if len(idx) else np.nan
        rows.append((val, Athr))
    return np.array(rows)

def run_parameter_robustness():
    D_vals = [80.0, 120.0, 160.0]
    rho_vals = [0.03, 0.05, 0.08]
    gamma_vals = [0.7, 1.0, 1.4]
    noise_vals = [0.1, 0.2, 0.3]

    rob_D = threshold_amplitude_for_param("D0", D_vals)
    rob_rho = threshold_amplitude_for_param("rho0", rho_vals)
    rob_gamma = threshold_amplitude_for_param("gamma", gamma_vals)
    rob_noise = threshold_amplitude_for_param("noise", noise_vals)

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

    axes[0,0].plot(rob_D[:,0], rob_D[:,1], marker="o")
    axes[0,0].set_title("Threshold vs baseline D")
    axes[0,0].set_xlabel("Baseline D")
    axes[0,0].set_ylabel("Threshold amplitude A*")

    axes[0,1].plot(rob_rho[:,0], rob_rho[:,1], marker="o")
    axes[0,1].set_title("Threshold vs baseline ρ")
    axes[0,1].set_xlabel("Baseline ρ")
    axes[0,1].set_ylabel("Threshold amplitude A*")

    axes[1,0].plot(rob_gamma[:,0], rob_gamma[:,1], marker="o")
    axes[1,0].set_title("Threshold vs drag γ")
    axes[1,0].set_xlabel("Drag γ")
    axes[1,0].set_ylabel("Threshold amplitude A*")

    axes[1,1].plot(rob_noise[:,0], rob_noise[:,1], marker="o")
    axes[1,1].set_title("Threshold vs noise")
    axes[1,1].set_xlabel("Noise amplitude")
    axes[1,1].set_ylabel("Threshold amplitude A*")

    plt.savefig(OUT / "reviewer_parameter_robustness.png", dpi=220)
    plt.close()

    robust_df = pd.DataFrame({
        "D_value": rob_D[:,0],
        "Astar_D": rob_D[:,1],
        "rho_value": rob_rho[:,0],
        "Astar_rho": rob_rho[:,1],
        "gamma_value": rob_gamma[:,0],
        "Astar_gamma": rob_gamma[:,1],
        "noise_value": rob_noise[:,0],
        "Astar_noise": rob_noise[:,1],
    })
    robust_df.to_csv(OUT / "reviewer_parameter_robustness.csv", index=False)

    return robust_df

# ============================================================
# 3) Scaling / nondimensionalization
# ============================================================
def logistic(x, x0, s):
    return 1.0 / (1.0 + np.exp(-(x - x0) / s))

def run_scaling_analysis():
    records = []
    D_grid = [80.0, 120.0, 160.0]
    F_grid = [2.0, 3.0, 4.0, 5.0]
    k_grid = [0.7, 1.0, 1.4]
    g_grid = [0.7, 1.0, 1.4]

    for D in D_grid:
        for F in F_grid:
            for k in k_grid:
                for g in g_grid:
                    Pi = ((D / 120.0) + 0.9 * (F / 3.0)) / (k * g)
                    closed = []
                    for s in range(8):
                        t, rwd = simulate_reduced_hybrid(
                            D0=D,
                            rho0=0.05,
                            k_adh=k,
                            gamma=g,
                            noise=0.2,
                            A=F / 3.0,
                            Tprime=48.0,
                            seed=5000 + s
                        )
                        closed.append(rwd[-1] >= 0.95)
                    records.append((D, F, k, g, Pi, np.mean(closed)))

    scale_df = pd.DataFrame(records, columns=["D", "Fext_proxy", "k_adh", "gamma", "Pi", "Pclose"])

    xdata = scale_df["Pi"].to_numpy()
    ydata = scale_df["Pclose"].to_numpy()

    popt, _ = curve_fit(logistic, xdata, ydata, p0=[1.8, 0.15], maxfev=10000)
    Pi_c, slope = popt

    xx = np.linspace(xdata.min(), xdata.max(), 300)
    yy = logistic(xx, Pi_c, slope)

    plt.figure(figsize=(7, 5))
    plt.scatter(scale_df["Pi"], scale_df["Pclose"], alpha=0.7, label="Simulation conditions")
    plt.plot(xx, yy, linewidth=2, label=f"Logistic fit (Πc ≈ {Pi_c:.2f})")
    plt.axvline(Pi_c, linestyle="--", label=f"Critical Π ≈ {Pi_c:.2f}")
    plt.xlabel(r"Drive-to-resistance ratio $\Pi$")
    plt.ylabel("Closure probability by 48 h")
    plt.title("Scaling collapse of wound closure outcomes")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "reviewer_scaling_analysis.png", dpi=220)
    plt.close()

    scale_df.to_csv(OUT / "reviewer_scaling_analysis.csv", index=False)
    return scale_df, Pi_c

# ============================================================
# Main execution
# ============================================================
if __name__ == "__main__":
    generate_schedule_figure()

    conv_dx, conv_ng = run_convergence_analysis()
    robust_df = run_parameter_robustness()
    scale_df, Pi_c = run_scaling_analysis()

Saved files:
hybrid_model_outputs/schedule_trajectories.png
hybrid_model_outputs/reviewer_convergence.png
hybrid_model_outputs/reviewer_convergence_dx.csv
hybrid_model_outputs/reviewer_convergence_agents.csv
hybrid_model_outputs/reviewer_parameter_robustness.png
hybrid_model_outputs/reviewer_parameter_robustness.csv
hybrid_model_outputs/reviewer_scaling_analysis.png
hybrid_model_outputs/reviewer_scaling_analysis.csv
hybrid_model_outputs/reviewer_convergence_robustness_scaling_summary.txt
